### Procesamiento de Lenguaje Natural I

# **Desafío 1**

### Alejandro Valle


### Vectorización de texto y modelo de clasificación Naïve Bayes con el dataset 20 newsgroups


In [1]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.naive_bayes import MultinomialNB, ComplementNB
from sklearn.metrics import f1_score

Utilizamos **20newsgroups** por ser un dataset clásico de NLP ya viene incluido y formateado en sklearn


In [2]:
from sklearn.datasets import fetch_20newsgroups
import numpy as np

## Carga de datos


Cargamos los datos (ya separados de forma predeterminada en train y test)


El dataset 20 Newsgroups contiene aproximadamente 18 000 publicaciones de grupos de noticias distribuidas en 20 temas. Está dividido en dos subconjuntos: uno para entrenamiento (train set) y otro para pruebas (test set).


In [ ]:
newsgroups_train = fetch_20newsgroups(subset="train", remove=("headers", "footers", "quotes"))
newsgroups_test = fetch_20newsgroups(subset="test", remove=("headers", "footers", "quotes"))

## Vectorización


Instanciamos un vectorizador.

Podemos ver diferentes parámetros de instanciación en la documentación de sklearn https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html


In [4]:
tfidfvect = TfidfVectorizer()

En el atributo `data` accedemos al texto


In [5]:
print(newsgroups_train.data[0])

I was wondering if anyone out there could enlighten me on this car I saw
the other day. It was a 2-door sports car, looked to be from the late 60s/
early 70s. It was called a Bricklin. The doors were really small. In addition,
the front bumper was separate from the rest of the body. This is 
all I know. If anyone can tellme a model name, engine specs, years
of production, where this car is made, history, or whatever info you
have on this funky looking car, please e-mail.


Con la interfaz habitual de sklearn podemos ajustar el vectorizador (obtener el vocabulario y calcular el vector IDF) y transformar directamente los datos.

Podemos denominar `X_train` como la matriz documento-término.


In [6]:
X_train = tfidfvect.fit_transform(newsgroups_train.data)

Recordemos que las vectorizaciones por conteos son de tipo sparse, por ello sklearn convenientemente devuelve los vectores de documentos como matrices de tipo sparse.


In [7]:
print(type(X_train))
print(f"shape: {X_train.shape}")
print(f"Cantidad de documentos: {X_train.shape[0]}")
print(f"Tamaño del vocabulario (dimensionalidad de los vectores): {X_train.shape[1]}")

<class 'scipy.sparse._csr.csr_matrix'>
shape: (11314, 101631)
Cantidad de documentos: 11314
Tamaño del vocabulario (dimensionalidad de los vectores): 101631


Una vez ajustado el vectorizador, podemos acceder a atributos como el vocabulario aprendido. Es un diccionario que va de términos a índices.

El índice es la posición en el vector de documento.


In [8]:
tfidfvect.vocabulary_["car"]

25775

Probamos con una palbra que no está en el documento.


In [9]:
# tfidfvect.vocabulary_["cocoliso"]

Es muy útil tener el diccionario opuesto que va de índices a términos


In [10]:
idx2word = {v: k for k, v in tfidfvect.vocabulary_.items()}

En `y_train` guardamos los targets que son enteros


In [11]:
y_train = newsgroups_train.target
y_train[:10]

array([ 7,  4,  4,  1, 14, 16, 13,  3,  2,  4])

Hay 20 clases correspondientes a los 20 grupos de noticias


In [12]:
print(f"clases {np.unique(newsgroups_test.target)}")
newsgroups_test.target_names

clases [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19]


['alt.atheism',
 'comp.graphics',
 'comp.os.ms-windows.misc',
 'comp.sys.ibm.pc.hardware',
 'comp.sys.mac.hardware',
 'comp.windows.x',
 'misc.forsale',
 'rec.autos',
 'rec.motorcycles',
 'rec.sport.baseball',
 'rec.sport.hockey',
 'sci.crypt',
 'sci.electronics',
 'sci.med',
 'sci.space',
 'soc.religion.christian',
 'talk.politics.guns',
 'talk.politics.mideast',
 'talk.politics.misc',
 'talk.religion.misc']

## Similaridad de documentos


Veamos similaridad de documentos. Tomemos algún documento


In [13]:
idx = 4811
print(newsgroups_train.data[idx])

THE WHITE HOUSE

                  Office of the Press Secretary
                   (Pittsburgh, Pennslyvania)
______________________________________________________________
For Immediate Release                         April 17, 1993     

             
                  RADIO ADDRESS TO THE NATION 
                        BY THE PRESIDENT
             
                Pittsburgh International Airport
                    Pittsburgh, Pennsylvania
             
             
10:06 A.M. EDT
             
             
             THE PRESIDENT:  Good morning.  My voice is coming to
you this morning through the facilities of the oldest radio
station in America, KDKA in Pittsburgh.  I'm visiting the city to
meet personally with citizens here to discuss my plans for jobs,
health care and the economy.  But I wanted first to do my weekly
broadcast with the American people. 
             
             I'm told this station first broadcast in 1920 when
it reported that year's presidential elec

Medimos la similaridad coseno con todos los documentos de train


In [14]:
cossim = cosine_similarity(X_train[idx], X_train)[0]

Podemos ver los valores de similaridad ordenados de mayor a menor


In [15]:
np.sort(cossim)[::-1]

array([1.        , 0.70930477, 0.67474953, ..., 0.        , 0.        ,
       0.        ], shape=(11314,))

Después vemos a qué documentos corresponden


In [16]:
np.argsort(cossim)[::-1]

array([ 4811,  6635,  4253, ...,  1534, 10055,  4750], shape=(11314,))

Obtenemos los 5 documentos más similares:


In [17]:
mostsim = np.argsort(cossim)[::-1][1:6]
print(mostsim)

[6635 4253 3596 4271 3746]


El documento original pertenece a la clase:


In [18]:
newsgroups_train.target_names[y_train[idx]]

'talk.politics.misc'

Revisamos las clases de los 5 más similares:


In [19]:
for i in mostsim:
    print(newsgroups_train.target_names[y_train[i]])

talk.politics.misc
talk.politics.misc
talk.politics.misc
talk.politics.misc
talk.politics.misc


### Modelo de clasificación Naïve Bayes


Instanciamos el modelo de clasificación Naive Bayes y lo entrenamos con sklearn


In [20]:
clf = MultinomialNB()
clf.fit(X_train, y_train)

,"alpha alpha: float or array-like of shape (n_features,), default=1.0Additive (Laplace/Lidstone) smoothing parameter(set alpha=0 and force_alpha=True, for no smoothing).",1.0
,"force_alpha force_alpha: bool, default=TrueIf False and alpha is less than 1e-10, it will set alpha to1e-10. If True, alpha will remain unchanged. This may causenumerical errors if alpha is too close to 0... versionadded:: 1.2.. versionchanged:: 1.4 The default value of `force_alpha` changed to `True`.",True
,"fit_prior fit_prior: bool, default=TrueWhether to learn class prior probabilities or not.If false, a uniform prior will be used.",True
,"class_prior class_prior: array-like of shape (n_classes,), default=NonePrior probabilities of the classes. If specified, the priors are notadjusted according to the data.",None
Name,Type,Value
"class_count_ class_count_: ndarray of shape (n_classes,)Number of samples encountered for each class during fitting. Thisvalue is weighted by the sample weight when provided.","ndarray[float64](20,)","[480.,584.,591.,...,564.,465.,377.]"
"class_log_prior_ class_log_prior_: ndarray of shape (n_classes,)Smoothed empirical log probability for each class.","ndarray[float64](20,)","[-3.16,-2.96,-2.95,...,-3. ,-3.19,-3.4 ]"
"classes_ classes_: ndarray of shape (n_classes,)Class labels known to the classifier","ndarray[int64](20,)","[ 0, 1, 2,...,17,18,19]"
"feature_count_ feature_count_: ndarray of shape (n_classes, n_features)Number of samples encountered for each (class, feature)during fitting. This value is weighted by the sample weight whenprovided.","ndarray[float64](20, 101631)","[[0. ,0.94,0. ,...,0. ,0. ,0. ], [1.39,0.6 ,0. ,...,0. ,0. ,0. ], [0.95,0.14,0. ,...,0. ,0. ,0. ], ..., [0.42,2.9 ,0.04,...,0. ,0. ,0. ], [0.61,1.36,0. ,...,0. ,0. ,0. ], [0.03,0.38,0. ,...,0. ,0. ,0. ]]"
"feature_log_prob_ feature_log_prob_: ndarray of shape (n_classes, n_features)Empirical log probability of featuresgiven a class, ``P(x_i|y)``.","ndarray[float64](20, 101631)","[[-11.56,-10.9 ,-11.56,...,-11.56,-11.56,-11.56], [-10.69,-11.1 ,-11.56,...,-11.56,-11.56,-11.56], [-10.9 ,-11.44,-11.57,...,-11.57,-11.57,-11.57], ..., [-11.22,-10.21,-11.54,...,-11.57,-11.57,-11.57], [-11.09,-10.7 ,-11.56,...,-11.56,-11.56,-11.56], [-11.53,-11.23,-11.56,...,-11.56,-11.56,-11.56]]"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,101631


Ya tenemos nuestro vectorizador ya ajustado en train, vectorizamos los textos
del conjunto de test.


In [21]:
X_test = tfidfvect.transform(newsgroups_test.data)
y_test = newsgroups_test.target
y_pred = clf.predict(X_test)

El F1-score es una métrica adecuada para evaluar el desempeño de modelos de clasificación, especialmente cuando existe desbalance entre clases.

- El promediado macro calcula el promedio del F1-score de cada clase, otorgando el mismo peso a todas las clases.
- El promediado micro calcula las métricas de forma global considerando todas las predicciones; en problemas de clasificación multiclase suele ser equivalente a la accuracy, por lo que no es la mejor métrica cuando el dataset está desbalanceado.


In [22]:
f1_score(y_test, y_pred, average="macro")

0.5854345727938506

---


## **Consigna del Desafío 1**

**Cada experimento realizado debe estar acompañado de una explicación o interpretación de lo observado.**


**1. Vectorizar documentos**

- Tomar 5 documentos al azar y medir similaridad con el resto de los documentos.
  Estudiar los 5 documentos más similares de cada uno analizar si tiene sentido
  la similaridad según el contenido del texto y la etiqueta de clasificación.

**2. Construir un modelo de clasificación por prototipos (tipo zero-shot).**

- Clasificar los documentos de un conjunto de test comparando cada uno con todos los de entrenamiento y asignar la clase al label del documento del conjunto de entrenamiento con mayor similaridad.

**3. Entrenar modelos de clasificación Naïve Bayes para maximizar el desempeño de clasificación**

- F1-Score Macro en el conjunto de datos de test. Considerar cambiar parámetros
  de instanciación del vectorizador y los modelos y probar modelos de Naïve Bayes Multinomial y ComplementNB.

**NO cambiar el hiperparámetro ngram_range de los vectorizadores**.

**4. Transponer la matriz documento-término.**

- De esa manera se obtiene una matriz término-documento que puede ser interpretada como una colección de vectorización de palabras.
- Estudiar ahora similaridad entre palabras tomando 5 palabras y estudiando sus 5 más similares.

**Elegir las palabras MANUALMENTE para evitar la aparición de términos poco interpretables**.


**1. Vectorizar documentos**

- Tomar 5 documentos al azar y medir similaridad con el resto de los documentos.
  Estudiar los 5 documentos más similares de cada uno analizar si tiene sentido
  la similaridad según el contenido del texto y la etiqueta de clasificación.


In [ ]:
import textwrap

rng = np.random.default_rng(42)
indexes = rng.choice(X_train.shape[0], size=5, replace=False)

N_CHARS = 600
WRAP_WIDTH = 90


def preview(text, n=N_CHARS, indent="     "):
    clean = " ".join(text.split())
    if not clean:
        return indent + "(documento vacío tras remover headers/footers/quotes)"
    clean = clean[:n] + ("..." if len(clean) > n else "")
    return textwrap.fill(clean, width=WRAP_WIDTH, initial_indent=indent, subsequent_indent=indent)


for index in indexes:
    cos_sim = cosine_similarity(X_train[index], X_train)[0]
    most_sim = np.argsort(cos_sim)[::-1][1:6]

    print(f"\n{'=' * 80}")
    print(f"Documento #{index}  |  Clase: {newsgroups_train.target_names[y_train[index]]}")
    print(f"{'-' * 80}")
    print(preview(newsgroups_train.data[index], indent=""), "\n")

    print("Documentos más similares:")
    for rank, doc in enumerate(most_sim, start=1):
        clase = newsgroups_train.target_names[y_train[doc]]
        print(f"  {rank}. Doc #{doc:<6} cos={cos_sim[doc]:.4f}  clase={clase}")
        print(preview(newsgroups_train.data[doc]), "\n")

    print()


Documento #8754  |  Clase: talk.religion.misc
--------------------------------------------------------------------------------
/(hudson) /If someone inflicts pain on themselves, whether they enjoy it or not, they /are
hurting themselves. They may be permanently damaging their body. That is true. It is also
none of your business. Some people may also reason that by reading the bible and being a
Xtian you are permanently damaging your brain. By your logic, it would be OK for them to
come into your home, take away your bible, and send you off to "re-education camps" to
save your mind from ruin. Are you ready for that? /(hudson) /And why is there nothing
wrong with it? Because you say so? Who gave you /the authority to sa... 

Documentos más similares:
  1. Doc #6552   cos=0.4904  clase=talk.religion.misc
     If I have a habit that I really want to break, and I am willing to make whatever
     sacrifice I need to make to break it, then I do so. There have been bad habits of
     mine tha

**Comentarios:**

Similitud coseno con TF-IDF parece funcionar adecuadamente en relacionar contexto de los documentos. Las clases aletorias de los documentos elegidos se correlacionan correctamente (en algunos casos) con documentos de la misma clase (aunque no necesariamente en contenido). Por ejemplo, el documento de la clase `comp.sys.mac.hardware` tuvo 2 de sus 5 vecinos totales de la misma clase, pero las clases restantes de diferente categoría son de la misma temática (puertos de impresoras).

La similaridad agrupa por vocabulario compartido, por lo tanto, es una buena aproximación léxica aunque no capture contexto semántico.


**2. Construir un modelo de clasificación por prototipos (tipo zero-shot).**

- Clasificar los documentos de un conjunto de test comparando cada uno con todos los de entrenamiento y asignar la clase al label del documento del conjunto de entrenamiento con mayor similaridad.


In [24]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import f1_score

knn = KNeighborsClassifier(n_neighbors=1, metric="cosine")
knn.fit(X_train, y_train)

knn_pred = knn.predict(X_test)

print(knn_pred)

f1_score(y_test, knn_pred, average="macro")

[ 0 19 17 ... 17 12 15]


0.5049911553681621

**Comentarios:**

Este es un enfoque sin entrenamiento `zero-shot` por que no se ajustan parámetros a partir de los datos. La consigna pide encontrar el documento con mayor similaridad, KNN es una buena aplicación para este problema con su métrica `cosine`. Se obtuvo un F1-Score menor que el NB de arriba. Además, es computacionalmente más costoso en inferencia: no hay entrenamiento pero cada predicción requiere comparar contra los 11314 documentos de entrenamiento mientras que NB solo compara contra 20 vectores de clase.


**3. Entrenar modelos de clasificación Naïve Bayes para maximizar el desempeño de clasificación**

- F1-Score Macro en el conjunto de datos de test. Considerar cambiar parámetros
  de instanciación del vectorizador y los modelos y probar modelos de Naïve Bayes Multinomial y ComplementNB.

  **NO cambiar el hiperparámetro ngram_range de los vectorizadores**.


**Baseline del NB original del notebook 0.5854345727938506**

**Baseline del zero-shot model 0.5049911553681621**

Estos son los valores a superar.


In [25]:
# Balance de las clases

import pandas as pd


def class_balance(y, target_names, name):
    counts = pd.Series(y).value_counts().sort_index()
    df = pd.DataFrame(
        {
            "clase": [target_names[i] for i in counts.index],
            "n_docs": counts.values,
            "proporcion": (counts.values / counts.sum()).round(4),
        }
    )
    df.insert(0, "set", name)
    return df


balance = pd.concat(
    [
        class_balance(y_train, newsgroups_train.target_names, "train"),
        class_balance(y_test, newsgroups_test.target_names, "test"),
    ]
)

balance.pivot(index="clase", columns="set", values=["n_docs", "proporcion"])

n_docs        proporcion        
set                        test  train       test   train
clase                                                    
alt.atheism               319.0  480.0     0.0424  0.0424
comp.graphics             389.0  584.0     0.0516  0.0516
comp.os.ms-windows.misc   394.0  591.0     0.0523  0.0522
comp.sys.ibm.pc.hardware  392.0  590.0     0.0520  0.0521
comp.sys.mac.hardware     385.0  578.0     0.0511  0.0511
comp.windows.x            395.0  593.0     0.0524  0.0524
misc.forsale              390.0  585.0     0.0518  0.0517
rec.autos                 396.0  594.0     0.0526  0.0525
rec.motorcycles           398.0  598.0     0.0528  0.0529
rec.sport.baseball        397.0  597.0     0.0527  0.0528
rec.sport.hockey          399.0  600.0     0.0530  0.0530
sci.crypt                 396.0  595.0     0.0526  0.0526
sci.electronics           393.0  591.0     0.0522  0.0522
sci.med                   396.0  594.0     0.0526  0.0525
sci.space                 394.0  593.0     0.0523  0.0524
soc.religion.christian    398.0  599.0     0.0528  0.0529
talk.politics.guns        364.0  546.0     0.0483  0.0483
talk.politics.mideast     376.0  564.0     0.0499  0.0498
talk.politics.misc        310.0  465.0     0.0412  0.0411
talk.religion.misc        251.0  377.0     0.0333  0.0333

ComplementNB suele usarse para datasets desbalaneados. No tenemos un dataset altamente desbalanceado, por lo que no se realizan tecnicas de balanceo.


Instancio un vectorizador con algunos parametros adicionales como experimentación. Estos documentos son textos de forums en el que muchas temáticas son similares entre sí, además que es muy probable que se use lenguaje coloquial. Hace sentido ignorar palabras raras y/o palabras que aparecen en todos los documentos.


In [26]:
tfidf_v2 = TfidfVectorizer(
    min_df=5,  # ignora palabras que aparecen en menos de 5 documentos
    max_df=0.9,  # ignora palabras que aparecen en más del 90% de los documentos
    stop_words="english",
    sublinear_tf=True,  # usa 1 + log(tf) en vez de tf lineal. Ayuda a amortiguar el efecto de palabras que se repiten muchas veces en un mismo documento
)

X_train_v2 = tfidf_v2.fit_transform(newsgroups_train.data)
X_test_v2 = tfidf_v2.transform(newsgroups_test.data)

print(f"Vocabulario nuevo: {X_train_v2.shape[1]} términos")
print(f"Vocabulario original: {X_train.shape[1]} términos")

Vocabulario nuevo: 17797 términos
Vocabulario original: 101631 términos


Pruebo con distintos valores de suavizado. Si una palabra nunca aparece en los docs de entrenamiento de una clase, la probabilidad estimada sería 0, lo cual rompe el cálculo para cualqueir documento de test que si contenga esa palabra. El valor por defecto es 1, es como "simular" de que si ha visto una palabra inexistente ya que un alpha 0 pues anula todo el producto.

Se experimentará con este hiperparámetro.


In [27]:
alphas = [0.001, 0.01, 0.1, 0.5, 1, 5]
models = {"MultinomialNB": MultinomialNB, "ComplementNB": ComplementNB}

results = []

for model_name, model_class in models.items():
    for alpha in alphas:
        clf = model_class(alpha=alpha)
        clf.fit(X_train_v2, y_train)
        y_pred = clf.predict(X_test_v2)
        f1 = f1_score(y_test, y_pred, average="macro")
        results.append({"modelo": model_name, "alpha": alpha, "f1_macro": f1})
        print(f"{model_name:15s} alpha={alpha:<6} F1-macro={f1:.4f}")

MultinomialNB   alpha=0.001  F1-macro=0.6408
MultinomialNB   alpha=0.01   F1-macro=0.6618
MultinomialNB   alpha=0.1    F1-macro=0.6750
MultinomialNB   alpha=0.5    F1-macro=0.6583
MultinomialNB   alpha=1      F1-macro=0.6461
MultinomialNB   alpha=5      F1-macro=0.6070
ComplementNB    alpha=0.001  F1-macro=0.6677
ComplementNB    alpha=0.01   F1-macro=0.6687
ComplementNB    alpha=0.1    F1-macro=0.6739
ComplementNB    alpha=0.5    F1-macro=0.6812
ComplementNB    alpha=1      F1-macro=0.6820
ComplementNB    alpha=5      F1-macro=0.6769


In [ ]:
results_df = pd.DataFrame(results).sort_values("f1_macro", ascending=False)
results_df

best = results_df.iloc[0]
print(f"Mejor modelo: {best['modelo']} (alpha={best['alpha']}) con F1-macro={best['f1_macro']:.4f}")

Mejor modelo: ComplementNB (alpha=1.0) con F1-macro=0.6820


In [ ]:
# Aislamos el efecto del vectorizador nuevo del efecto del tuneo de alpha/modelo
# usando alpha=1 (default) como referencia común en ambos vectorizadores

clf_baseline = MultinomialNB()  # alpha=1 default
clf_baseline.fit(X_train, y_train)
f1_baseline = f1_score(y_test, clf_baseline.predict(X_test), average="macro")

f1_vectorizer_only = results_df.loc[
    (results_df["modelo"] == "MultinomialNB") & (results_df["alpha"] == 1), "f1_macro"
].iloc[0]

f1_full_tuning = results_df.iloc[0]["f1_macro"]

print(f"Baseline (vectorizador original, alpha=1 default):      F1-macro = {f1_baseline:.4f}")
print(
    f"Solo vectorizador nuevo (alpha=1 default):               F1-macro = {f1_vectorizer_only:.4f}"
)
print(
    f"Vectorizador nuevo + mejor alpha/modelo ({best['modelo']}, alpha={best['alpha']}): "
    f"F1-macro = {f1_full_tuning:.4f}"
)
print()
print(f"Mejora atribuible al vectorizador:          {f1_vectorizer_only - f1_baseline:+.4f}")
print(f"Mejora atribuible al tuneo de alpha/modelo: {f1_full_tuning - f1_vectorizer_only:+.4f}")

Baseline (vectorizador original, alpha=1 default):      F1-macro = 0.5854
Solo vectorizador nuevo (alpha=1 default):               F1-macro = 0.6461
Vectorizador nuevo + mejor alpha/modelo (ComplementNB, alpha=1.0): F1-macro = 0.6820

Mejora atribuible al vectorizador:          +0.0607
Mejora atribuible al tuneo de alpha/modelo: +0.0359


**Comentarios:**

- Vectorizer: Agregando unos params mas respecto a la configuración default hecha más arriba se redujo el vocabulario por aproximadamente un 82%. De 101631 a 17797. A pesar de tener muchisima menos dimensionalidad, el F1-macro mejoró respecto al baseline original hasta un máximo de 0.682. Esto sugiere que gran parte de ese vocabulario reducido eran ruido (términos raros, comunes, repetitivos, etc)
- Alpha: el comportamiento de alpha en MultiNomialNb no es monótono. Se llega a un pico en el valor default. Por el contrario, ComplementNB parece menos sensible al hiperpárametro `alpha` ya que la variación entre el peor aplha y el mejor es mucho menor que en MultiNomialNB. En MultinomialNB utiliza conteos por clase individual el cual puede ser pequeño, en ComplementNB se utiliza los conteos agregados de casi todo el corpus, el cual puede ser mucho más grande (Estos conteos estan en el denominador de la ecuación junto con el alpha).
- Conclusiones: Con la experimentación en la anterior celda se le atribuye la mejora en F1-score a ambos ajustes hechos: el ajuste de parametros del vectorizador y ComplementNB


**4. Transponer la matriz documento-término.**

- De esa manera se obtiene una matriz término-documento que puede ser interpretada como una colección de vectorización de palabras.
- Estudiar ahora similaridad entre palabras tomando 5 palabras y estudiando sus 5 más similares.

**Elegir las palabras MANUALMENTE para evitar la aparición de términos poco interpretables**.


Utilizo el dataset creado para el desafio, con el vocabulario reducido
También el dataset original para comparación


In [30]:
idx2word_v2 = {v: k for k, v in tfidf_v2.vocabulary_.items()}


X_train_T = X_train.T.tocsr()
X_train_v2_T = X_train_v2.T.tocsr()

print(f"Matriz termino-documento (vocab original): {X_train_T.shape}")
print(f"Matriz termino-documento (vocab reducido):  {X_train_v2_T.shape}")

Matriz termino-documento (vocab original): (101631, 11314)
Matriz termino-documento (vocab reducido):  (17797, 11314)


In [ ]:
def most_similar_words(word, vocabulary, X_T, idx2word, top_n=5):
    if word not in vocabulary:
        return None
    idx = vocabulary[word]
    sims = cosine_similarity(X_T[idx], X_T)[0]
    most_sim_idx = np.argsort(sims)[::-1][1 : top_n + 1]
    return [(idx2word[i], round(float(sims[i]), 4)) for i in most_sim_idx]


words = ["car", "linux", "football", "space", "government"]

for word in words:
    print(f"\n{'=' * 60}")
    print(f"Palabra: '{word}'")
    print(f"{'-' * 60}")

    sim_orig = most_similar_words(word, tfidfvect.vocabulary_, X_train_T, idx2word)
    sim_v2 = most_similar_words(word, tfidf_v2.vocabulary_, X_train_v2_T, idx2word_v2)

    print("Vocabulario original:")
    if sim_orig is None:
        print("  (palabra no esta en el vocabulario original)")
    else:
        for w, sim in sim_orig:
            print(f"  {w:<20} cos={sim}")

    print("\nVocabulario reducido:")
    if sim_v2 is None:
        print("  (palabra no esta en el vocabulario reducido)")
    else:
        for w, sim in sim_v2:
            print(f"  {w:<20} cos={sim}")


Palabra: 'car'
------------------------------------------------------------
Vocabulario original:
  cars                 cos=0.1797
  criterium            cos=0.177
  civic                cos=0.1748
  owner                cos=0.1689
  dealer               cos=0.1681

Vocabulario reducido:
  cars                 cos=0.1952
  dealer               cos=0.186
  civic                cos=0.1662
  owner                cos=0.1545
  engine               cos=0.1439

Palabra: 'linux'
------------------------------------------------------------
Vocabulario original:
  386bsd               cos=0.5133
  distributing         cos=0.3009
  libs                 cos=0.2984
  dpg                  cos=0.2946
  f2c                  cos=0.261

Vocabulario reducido:
  386bsd               cos=0.5185
  libs                 cos=0.2843
  distributing         cos=0.2363
  sysv                 cos=0.2103
  adopted              cos=0.1935

Palabra: 'football'
--------------------------------------------------------

**Comentarios:**

Con las matrices transpuestas se comparan filas de palabras entre sí respecto a los documentos en los que aparecen. Dos palabras tendrán vectores parecidos si tienden a co-ocurrir de forma consistente a través de muchos documentos, con pesos TF-IDF similares

Se obtuvieron resultados interesantes con el vocabulario reducido, pero mixtos con el vocabulario original: con este último, palabras como `the` y `of` aparecieron entre las más similares a `government`, mientras que con el vocabulario reducido ese comportamiento desapareció. Esto tiene sentido porque `the` y `of` son stop words que aparecen en prácticamente todos los documentos del corpus, su patrón de co-ocurrencia termina pareciéndose al de cualquier palabra muy frecuente. En el vocabulario reducido, se eliminan las stop words por lo que esta similitud desaparece
